# Picotron: native Triton pretraining demo

This short Kaggle demo trains a native Picotron decoder from scratch on FineWeb-Edu, verifies the real Triton RMSNorm and SwiGLU paths, saves safetensors weights, and runs inference.

> **Scope:** pretraining is the stable demo path. Picotron also includes experimental SFT support, but it is intentionally not part of this submission notebook.


## 1. Setup

Clone the repository and install the current package. Enable a Kaggle GPU accelerator first.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path('/kaggle/working/picotron')
REPO_URL = 'https://github.com/AndyMLAndAI/picotron.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO, check=True)
os.chdir(REPO)

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running this notebook.'
DEVICE = torch.device('cuda:0')
print('GPU:', torch.cuda.get_device_name(0))
print('compute capability:', torch.cuda.get_device_capability(0))


## 2. Write the native pretraining configuration

The tied 50,257 x 20 token embedding/output table makes this roughly a 1M-parameter decoder. The cache contains 100M FineWeb-Edu tokens; this demo trains 1,000 steps (about 8.2M token positions).


In [ ]:
PRETRAIN_CONFIG = REPO / 'config_native_1m_fineweb.yaml'
PRETRAIN_CHECKPOINT = REPO / 'artifacts/native_1m_fineweb/model.safetensors'
PRETRAIN_CONFIG.write_text('''
checkpoints:
  checkpoint_interval: 500
  checkpoints_path: artifacts/native_1m_fineweb/model.safetensors
  save_final_state: true

model:
  dtype: float16
  compile_model: false
  triton_kernels:
    rmsnorm: true
    swiglu: true
    rope: false
    attention: false
    cross_entropy: false
    adamw: false
  model_config:
    vocab_size: 50257
    hidden_size: 20
    intermediate_size: 80
    num_hidden_layers: 8
    num_attention_heads: 2
    num_key_value_heads: 1
    attention_type: gqa
    rope_theta: 1000000.0
    nope_layers: [3]
    sliding_window_size: 128
    moe_config: null
    kv_lora_rank: null
    tie_word_embeddings: true
    position_embedding_type: rope
    gradient_checkpointing: false

optimizer:
  learning_rate_scheduler:
    learning_rate: 0.0003
    lr_decay_style: constant
  optimizer_factory:
    name: adamw
    adam_beta1: 0.9
    adam_beta2: 0.95
    adam_eps: 1.0e-8
  weight_decay: 0.1
  clip_grad: 1.0

parallelism:
  dp: 1
  zero_stage: 0

tokens:
  sequence_length: 256
  micro_batch_size: 32
  train_steps: 1000

data:
  tokenizer_name: gpt2
  vocab_size: 50257
  token_cache_dir: data/token_cache
  datasets:
    - hf_name: HuggingFaceFW/fineweb-edu
      hf_config: sample-10BT
      target_tokens: 100000000
      text_field: text
      weight: 1.0
  num_workers: 4
  prefetch_factor: 2

logging:
  iteration_step_info_interval: 100
  file_logging: true
  file_logging_output_dir: logs

general:
  project: picotron
  run: native_1m_fineweb_triton
  seed: 1337
'''.lstrip(), encoding='utf-8')
print(PRETRAIN_CONFIG.read_text())


## 3. Verify Triton, train, and save a safetensors checkpoint

The probe runs Triton RMSNorm and SwiGLU forward/backward before the long job. It stops immediately if either path falls back.


In [ ]:
from picotron.config import load_config
from picotron.models.picotron_decoder import PicotronDecoderModel, RMSNorm
from picotron.nn.feedforward import SwiGLU
from picotron.utils.hardware import detect_triton_support

triton_report = detect_triton_support(enabled=True, device=DEVICE)
print('Triton preflight:', triton_report)
assert triton_report.installed and triton_report.hardware_compatible

probe_config = load_config(PRETRAIN_CONFIG)
probe_model = PicotronDecoderModel(probe_config).to(DEVICE).train()
probe_ids = torch.randint(0, probe_config.model.model_config.vocab_size, (1, 16), device=DEVICE)
with torch.autocast(device_type='cuda', dtype=torch.float16):
    probe_loss = probe_model(probe_ids).float().mean()
probe_loss.backward()
fallback_modules = [module.__class__.__name__ for module in probe_model.modules()
                    if isinstance(module, (RMSNorm, SwiGLU)) and module._triton_fallback_warned]
assert not fallback_modules, f'Triton probe fell back to PyTorch in: {fallback_modules}'
print('Triton probe passed: RMSNorm + SwiGLU forward/backward executed without fallback.')
del probe_model, probe_loss
torch.cuda.empty_cache()

subprocess.run(['picotron', '--config', str(PRETRAIN_CONFIG)], cwd=REPO, check=True)
assert PRETRAIN_CHECKPOINT.exists(), f'Missing final checkpoint: {PRETRAIN_CHECKPOINT}'
print('Native pretraining checkpoint:', PRETRAIN_CHECKPOINT)


## 4. Native inference after pretraining

The native helper intentionally recomputes context per generated token because KV caching is not implemented yet.


In [ ]:
from transformers import AutoTokenizer
from picotron.serialize import load_native_model

native_model = load_native_model(PRETRAIN_CHECKPOINT, device=DEVICE).eval()
native_tokenizer = AutoTokenizer.from_pretrained('gpt2', use_fast=True)
if native_tokenizer.pad_token_id is None:
    native_tokenizer.pad_token = native_tokenizer.eos_token

@torch.inference_mode()
def generate_native(prompt: str, max_new_tokens: int = 64) -> str:
    token_ids = native_tokenizer.encode(prompt, add_special_tokens=False)
    for _ in range(max_new_tokens):
        context = torch.tensor([token_ids[-256:]], dtype=torch.long, device=DEVICE)
        next_token = native_model(context)[0, -1].argmax().item()
        token_ids.append(next_token)
        if next_token == native_tokenizer.eos_token_id:
            break
    return native_tokenizer.decode(token_ids, skip_special_tokens=True)

for prompt in ('The future of machine learning is', 'Python functions are useful because'):
    print(f'\nPROMPT: {prompt}\n{generate_native(prompt)}')


## Experimental post-training

Picotron also exposes experimental full SFT APIs for compatible Hugging Face causal language models. This release demo deliberately focuses on the verified native pretraining path.
